#  Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib        # Saves Python objects to disk (scalers, models)
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler     # Scales features to mean=0, std=1
from sklearn.model_selection import train_test_split # Splits data into train/test sets
from imblearn.over_sampling import SMOTE             # Creates synthetic fraud samples

os.makedirs('../models', exist_ok=True)  # Create models/ folder if not exists
print(' Imports done!')

 Imports done!


# Load data

In [4]:
# Load the clean CSV saved at end of NB1
df = pd.read_csv('creditcard_clean.csv')

print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

Loaded: 284,807 rows × 31 columns


# Create new features

In [5]:
# Feature 1: Hour of day (0-23)
# Time column = seconds since first transaction
# % 86400 = remainder after full days, /3600 = convert to hours
df['hour_of_day'] = (df['Time'] / 3600 % 24).astype(int)

# Feature 2: Is Night (fraud spikes between midnight–5am)
# Creates 1 if hour is 0–5, else 0
df['is_night'] = ((df['hour_of_day'] >= 0) & (df['hour_of_day'] <= 5)).astype(int)

# Feature 3: Log of Amount
# log1p = log(1+x), avoids error when Amount=0
# Shrinks large values so model isn't biased by $50,000 outliers
df['amount_log'] = np.log1p(df['Amount'])

# Feature 4: Amount category bucket (tiny/small/medium/large/huge)
df['amount_bin'] = pd.cut(
    df['Amount'],
    bins=[0, 10, 50, 200, 1000, 50000],
    labels=[0, 1, 2, 3, 4]
)
df['amount_bin'] = df['amount_bin'].astype(float).fillna(0)

print('New features created:')
print(f'  hour_of_day : {df["hour_of_day"].min()}–{df["hour_of_day"].max()}')
print(f'  is_night    : {df["is_night"].sum():,} night transactions')
print(f'  amount_log  : min={df["amount_log"].min():.2f}, max={df["amount_log"].max():.2f}')
print(f'  amount_bin  : 0–4 bucket')
print(f'\nTotal features now: {df.shape[1]}')

New features created:
  hour_of_day : 0–23
  is_night    : 23,934 night transactions
  amount_log  : min=0.00, max=10.15
  amount_bin  : 0–4 bucket

Total features now: 35


# Verify new features

In [6]:
# Quick check — make sure new features look right
df[['Time', 'Amount', 'hour_of_day', 'is_night',
    'amount_log', 'amount_bin', 'Class']].head(10)

,Time,Amount,hour_of_day,is_night,amount_log,amount_bin,Class
0,0.0,149.62,0,1,5.014760,2.0,0
1,0.0,2.69,0,1,1.305626,0.0,0
2,1.0,378.66,0,1,5.939276,3.0,0
3,1.0,123.50,0,1,4.824306,2.0,0
4,2.0,69.99,0,1,4.262539,2.0,0
5,2.0,3.67,0,1,1.541159,0.0,0
6,4.0,4.99,0,1,1.790091,0.0,0
7,7.0,40.80,0,1,3.732896,1.0,0
8,7.0,93.20,0,1,4.545420,2.0,0
9,9.0,3.68,0,1,1.543298,0.0,0


# Define X and y

In [7]:
# X = input features (everything the model sees)
# y = target (what the model predicts: 0=legit, 1=fraud)
# Drop 'Class' (that's y) and 'Time' (replaced by hour_of_day)
X = df.drop(columns=['Class', 'Time'])
y = df['Class']

print(f'Features X: {X.shape}')   # Should be (284807, ~31)
print(f'Target   y: {y.shape}')
print(f'\nFeature list: {list(X.columns)}')

Features X: (284807, 33)
Target   y: (284807,)

Feature list: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'hour_of_day', 'is_night', 'amount_log', 'amount_bin']


# Scale features

In [8]:
# StandardScaler transforms data to: mean=0, standard deviation=1
# Formula: z = (x - mean) / std
# WHY: ML models treat big numbers as more important unless we scale
# V1–V28 are already PCA-scaled; we only scale Amount and new features

cols_to_scale = ['Amount', 'amount_log', 'hour_of_day']

scaler = StandardScaler()
# fit_transform: 1) learns mean & std, 2) applies transformation
X[cols_to_scale] = scaler.fit_transform(X[cols_to_scale])

# MUST save scaler — prediction time needs the SAME scaler!
joblib.dump(scaler, '../models/scaler.pkl')

print('✅ Scaling done!')
print(f'Amount mean after scaling : {X["Amount"].mean():.4f}  (should be ~0)')
print(f'Amount std  after scaling : {X["Amount"].std():.4f}   (should be ~1)')
print('💾 Scaler saved to models/scaler.pkl')

✅ Scaling done!
Amount mean after scaling : -0.0000  (should be ~0)
Amount std  after scaling : 1.0000   (should be ~1)
💾 Scaler saved to models/scaler.pkl


# Train / Val / Test split

In [9]:
# Split 1: 80% train+val, 20% test
# stratify=y ensures same fraud % in both splits (very important!)
# random_state=42 makes split reproducible every run
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Split 2: from the 80%, take 10% as validation
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.125,   # 0.125 × 80% = 10% of total
    random_state=42,
    stratify=y_temp
)

print('Split Summary:')
print(f'  Train      : {X_train.shape[0]:,} rows')
print(f'  Validation : {X_val.shape[0]:,} rows')
print(f'  Test       : {X_test.shape[0]:,} rows')
print(f'\nFraud % in each split:')
print(f'  Train : {y_train.mean()*100:.3f}%')
print(f'  Val   : {y_val.mean()*100:.3f}%')
print(f'  Test  : {y_test.mean()*100:.3f}%')

Split Summary:
  Train      : 199,364 rows
  Validation : 28,481 rows
  Test       : 56,962 rows

Fraud % in each split:
  Train : 0.173%
  Val   : 0.172%
  Test  : 0.172%


#  SMOTE oversampling

In [10]:
# SMOTE creates SYNTHETIC fraud examples by:
# 1. Taking a real fraud sample
# 2. Finding its 5 nearest fraud neighbors
# 3. Creating new fake samples between them
#  Apply SMOTE ONLY on training data — NEVER on val or test!

print('Before SMOTE:')
print(f'  Legit : {(y_train==0).sum():,}')
print(f'  Fraud : {(y_train==1).sum():,}')

smote = SMOTE(
    sampling_strategy=0.1,  # After SMOTE, fraud = 10% of legit count
    random_state=42,
    k_neighbors=5
)

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print('\nAfter SMOTE:')
print(f'  Legit : {(y_train_sm==0).sum():,}')
print(f'  Fraud : {(y_train_sm==1).sum():,}')
print(f'  Total : {len(X_train_sm):,}')

Before SMOTE:
  Legit : 199,019
  Fraud : 345

After SMOTE:
  Legit : 199,019
  Fraud : 19,901
  Total : 218,920


 # Save all splits

In [11]:
# Save everything in one file so NB3 can load it easily
joblib.dump({
    'X_train'    : X_train,
    'y_train'    : y_train,
    'X_train_sm' : X_train_sm,   # SMOTE version
    'y_train_sm' : y_train_sm,
    'X_val'      : X_val,
    'y_val'      : y_val,
    'X_test'     : X_test,
    'y_test'     : y_test,
    'feature_names': list(X.columns)
}, '../models/data_splits.pkl')

print(' All splits saved to models/data_splits.pkl')
print('  Now open NB3!')

 All splits saved to models/data_splits.pkl
  Now open NB3!
